### The goal of this problem is to optimize the parameters and hyperparameters of a pipeline.

## Dataset to use

URL = "https://raw.githubusercontent.com/mdogy/dataForEng1999/master/pi_diabetes.csv"

In [81]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, KFold, RandomizedSearchCV
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline

In [14]:
dataset_url = "https://raw.githubusercontent.com/mdogy/dataForEng1999/master/pi_diabetes.csv"
df_pi_diabetes = pd.read_csv(dataset_url)
df_pi_diabetes.shape

(768, 9)

In [15]:
df_pi_diabetes.head()


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [9]:
df_pi_diabetes.isna().sum()

,0
Pregnancies,0
Glucose,0
BloodPressure,0
SkinThickness,0
Insulin,0
BMI,0
DiabetesPedigreeFunction,0
Age,0
Outcome,0


no missing values. datasen seems to be clean so we dont have to deal with imputing data

In [12]:
df_pi_diabetes.describe()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
count,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000,768.000000
mean,3.845052,120.894531,69.105469,20.536458,79.799479,31.992578,0.471876,33.240885,0.348958
std,3.369578,31.972618,19.355807,15.952218,115.244002,7.884160,0.331329,11.760232,0.476951
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.078000,21.000000,0.000000
25%,1.000000,99.000000,62.000000,0.000000,0.000000,27.300000,0.243750,24.000000,0.000000
50%,3.000000,117.000000,72.000000,23.000000,30.500000,32.000000,0.372500,29.000000,0.000000
75%,6.000000,140.250000,80.000000,32.000000,127.250000,36.600000,0.626250,41.000000,1.000000
max,17.000000,199.000000,122.000000,99.000000,846.000000,67.100000,2.420000,81.000000,1.000000


as we can see that Insuling column needs to be preprocessed

In [17]:
df_pi_diabetes.columns

Index(['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin',
       'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
      dtype='object')

In [41]:
target_variable = "Outcome"
df_input_vars = df_pi_diabetes[[col for col in df_pi_diabetes.columns if col != target_variable]].values
df_output_var = df_pi_diabetes[target_variable].values
X_train, X_test, y_train, y_test = train_test_split(df_input_vars, df_output_var, test_size=0.3, random_state=123, stratify=df_output_var)
print(f" Train shape: X={X_train.shape} & y={y_train.shape} \n Test shape: X={X_test.shape} & y={y_test.shape}")

 Train shape: X=(537, 8) & y=(537,) 
 Test shape: X=(231, 8) & y=(231,)


In [88]:
def print_metrics(y_test, y_pred):
  accuracy = accuracy_score(y_test, y_pred)
  recall = recall_score(y_test, y_pred)
  precision = precision_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred)
  conf_mat = confusion_matrix(y_test, y_pred)
  print(f"Confusion matrix: \n{conf_mat}\n")
  print(f'Accuracy: {accuracy:.4f}')
  print(f'Recall: {recall:.4f}')
  print(f'Precision: {precision:.4f}')
  print(f'F1: {f1:.4f}')

In [89]:
logreg_basic = LogisticRegression(max_iter=1000)
logreg_basic.fit(X_train, y_train)
y_pred = logreg_basic.predict(X_test)
print("Logistic Regression baseline: no hyperparameter tune and no preprocessing")
print_metrics(y_test, y_pred)

Logistic Regression baseline: no hyperparameter tune and no preprocessing
Confusion matrix: 
[[132  18]
 [ 38  43]]

Accuracy: 0.7576
Recall: 0.5309
Precision: 0.7049
F1: 0.6056


In [90]:
# with cross validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(logreg_basic, X_train, y_train, cv=kf)
print(cv_scores.mean())

0.7690031152647975


In [91]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [92]:
logreg_basic = LogisticRegression()
logreg_basic.fit(X_train_scaled, y_train)
y_pred = logreg_basic.predict(X_test_scaled)

print("Logistic Regression with preprocessing")
print_metrics(y_test, y_pred)

Logistic Regression with preprocessing
Confusion matrix: 
[[132  18]
 [ 39  42]]

Accuracy: 0.7532
Recall: 0.5185
Precision: 0.7000
F1: 0.5957


In [100]:
# now modelling with SVM
print("SVM model with hyperparams search using RandomizedSearchCV")
svm_kernels = ["linear", "poly", "rbf", "sigmoid"]

kf = KFold(n_splits=5, shuffle=True, random_state=42)
params = {
    "svm__C": np.linspace(0.1, 1.0, 20),
    'svm__kernel': svm_kernels,
}
pipe_steps= [
    ("scaler", StandardScaler()),
    ("svm",SVC())
]
pipeline = Pipeline(pipe_steps)
random_search = RandomizedSearchCV(pipeline, params, n_iter=40, cv=kf)

random_search.fit(X_train, y_train)


SVM model with hyperparams search using RandomizedSearchCV


RandomizedSearchCV(cv=KFold(n_splits=5, random_state=42, shuffle=True),
                   estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                             ('svm', SVC())]),
                   n_iter=40,
                   param_distributions={'svm__C': array([0.1       , 0.14736842, 0.19473684, 0.24210526, 0.28947368,
       0.33684211, 0.38421053, 0.43157895, 0.47894737, 0.52631579,
       0.57368421, 0.62105263, 0.66842105, 0.71578947, 0.76315789,
       0.81052632, 0.85789474, 0.90526316, 0.95263158, 1.        ]),
                                        'svm__kernel': ['linear', 'poly', 'rbf',
                                                        'sigmoid']})

In [101]:

print(f"Best Params: {random_search.best_params_}, best score {random_search.best_score_}")
best_model = random_search.best_estimator_
y_pred = best_model.predict(X_test)

print_metrics(y_test, y_pred)

Best Params: {'svm__kernel': 'sigmoid', 'svm__C': np.float64(0.19473684210526315)}, best score 0.7819833852544132
Confusion matrix: 
[[131  19]
 [ 45  36]]

Accuracy: 0.7229
Recall: 0.4444
Precision: 0.6545
F1: 0.5294


In [112]:
models = {"logreg": LogisticRegression(max_iter=1000), "svm": SVC()}
model_params = {
    "logreg": {
      "logreg__solver": ["newton-cg", "saga", "lbfgs"],
      "logreg__C": np.linspace(0.001, 1.0, 20)
    },
    "svm": {
      "svm__C": np.linspace(0.1, 1.0, 20),
      'svm__kernel': svm_kernels,
    }
}
results = {}

for model_name, model in models.items():
  kf = KFold(n_splits=5, random_state=42, shuffle=True)
  cv_scores = cross_val_score(model, X_train, y_train, cv=kf)
  results[model_name] = cv_scores

  pipe_steps= [
      ("scaler", StandardScaler()),
      (model_name, model)
  ]
  pipeline = Pipeline(pipe_steps)
  params = model_params[model_name]
  random_search = RandomizedSearchCV(pipeline, params, n_iter=40, cv=kf)

  random_search.fit(X_train, y_train)
  print(f"{model_name} \n Best Params: {random_search.best_params_}, best score {random_search.best_score_}")
  best_model = random_search.best_estimator_
  y_pred = best_model.predict(X_test)

  print_metrics(y_test, y_pred)
  print("\n\n")


logreg 
 Best Params: {'logreg__solver': 'lbfgs', 'logreg__C': np.float64(0.3164736842105263)}, best score 0.7746105919003116
Confusion matrix: 
[[132  18]
 [ 40  41]]

Accuracy: 0.7489
Recall: 0.5062
Precision: 0.6949
F1: 0.5857



svm 
 Best Params: {'svm__kernel': 'linear', 'svm__C': np.float64(0.4789473684210527)}, best score 0.7764278296988577
Confusion matrix: 
[[130  20]
 [ 39  42]]

Accuracy: 0.7446
Recall: 0.5185
Precision: 0.6774
F1: 0.5874





Based on this research above parameters are best for Logistric Regression and SVM.  
However, baseline logistic regression got best F1 score 0.60